In [1]:
"""本地测试脚本 - 不依赖 FastAPI"""
import os
import sys
from dotenv import load_dotenv
from openai import OpenAI

# 添加项目根目录到 Python 路径
# 获取项目根目录（tests 目录的父目录）
project_root = os.path.dirname(os.getcwd())
print(f'项目根目录: {project_root}')
if project_root not in sys.path:
    sys.path.insert(0, project_root)

# 从 rag 模块统一导入
from rag import (
    build_vector_db,
    hybrid_search,
    dense_search,
    sparse_search,
    get_doc_count,
    resolve_placeholders,
    router,
    rewrite,
    intent,
    rewrite_for_retrieval,
)

load_dotenv()

项目根目录: d:\github\rag_project\rag_raw\rag_new


d:\anaconda3\envs\tf2\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


True

In [2]:

# ============ 配置 ============
REPORT_PATH =  "D:/github/rag_project/rag_财报/财报" 
API_KEY = os.getenv('DEEPSEEK_API_KEY')
API_BASE = os.getenv('DEEPSEEK_API_BASE')
MODEL = os.getenv('DEEPSEEK_MODEL')

client = OpenAI(api_key=API_KEY, base_url=API_BASE)

model = None
history = []  # 对话历史

def init_system():
    """初始化系统"""
    global model, history
    
    # 初始化向量数据库
    model = build_vector_db(REPORT_PATH, rebuild=False)
    
    # 系统提示词
    system_prompt = """你是一个专业的财务分析助手。\n
1. 请**仅根据**【本轮财报片段】来回答用户的问题。\n
2. 如果提供的片段中没有相关信息，请直接回答"根据提供的财报内容，我无法回答该问题"，绝不要编造数据。\n
3. 在回答的时候, 不需要输出分析的过程, 直接摆出关键数据然后回答用户问题\n
4. 不要开启思考模式。尽可能少消费token完成任务。"""
    
    # 清空历史并添加系统提示词
    history = [{"role": "system", "content": system_prompt}]
    
    # 显示状态
    doc_count = get_doc_count()
    print(f"[2/2] 初始化完成！")
    print(f"   - 当前知识库文本块数: {doc_count}")
    print(f"   - 财报文件: {REPORT_PATH}")
    print("=" * 60)


def test_search(query: str, mode: str = "hybrid", top_k: int = 5, show_sources: bool = True, source_filter: str = None):
    """
    测试检索功能

    Args:
        query: 查询问题
        mode: 检索模式 (dense/sparse/hybrid)
        top_k: 返回结果数
        show_sources: 是否显示检索到的原始片段
        source_filter: 指定检索的源文件
    """
    print(f"\n{'='*60}")
    print(f" 查询问题: {query}")
    print(f" 检索模式: {mode}")
    print(f" 返回数量: {top_k}")
    print("=" * 60)

    # 根据mode执行不同的检索
    if mode == "dense":
        results = dense_search(query, top_k=top_k, source_filter=source_filter)
        print(" 向量检索 (BGE向量)")
    elif mode == "sparse":
        results = sparse_search(query, top_k=top_k, source_filter=source_filter)
        print(" 关键词检索 (BM25)")
    else:  # hybrid
        results = hybrid_search(query, top_k=top_k, source_filter=source_filter)
        # 同时获取密集检索结果用于显示得分
        dense_results = dense_search(query, top_k=10, source_filter=source_filter)
        print(" 混合检索 (BGE向量 + BM25 + RRF)")

    # 显示检索结果
    print(f"\n 检索到 {len(results)} 条相关片段:")
    print("-" * 60)

    for i, (doc_id, chunk_text, score, source_file) in enumerate(results, 1):
            if mode == "dense":
                # 将距离转换为相似度：similarity = 1 / (1 + distance)
                similarity = 1 / (1 + score)
                print(f"\n[片段 {i}] ID: {doc_id} | 来源: {source_file}")
            elif mode == "sparse":
                print(f"\n[片段 {i}] ID: {doc_id} | 来源: {source_file}")
                print(f"  BM25得分: {score:.4f}")
            else:  # hybrid
                d_score = {d_id: s for d_id, _, s, _ in dense_results}.get(doc_id, float('inf'))
                similarity = 1 / (1 + d_score) if d_score != float('inf') else 0
                print(f"\n[片段 {i}] ID: {doc_id} | 来源: {source_file}")
                print(f"  RRF得分: {score:.4f} | 向量相似度: {similarity:.4f}")
            
            if show_sources:
                print(f"  内容: {chunk_text}")
    return results


def test_generation(query: str, mode: str = "hybrid", top_k: int = 5):
    """
    测试完整流程（意图识别 + 检索 + 生成）

    Args:
        query: 查询问题
        mode: 检索模式 (dense/sparse/hybrid)
        top_k: 检索返回数量
    """
    global history, last_retrieved_chunks
    
    # --- 1. 意图识别 ---
    user_intent = intent(query, history)
    
    if user_intent == "chat":
        # 闲聊：直接调用LLM回答
        print("\n检测到闲聊，直接回答...")
        
        # 保留系统提示词，添加用户问题
        chat_history = [history[0]] if history and history[0].get('role') == 'system' else []
        chat_history.append({"role": "user", "content": query})
        
        response = client.chat.completions.create(
            model=MODEL,
            extra_body={"thinking": {"type": "disabled"}},
            messages=chat_history,
            temperature=0.7
        )
        answer = response.choices[0].message.content
        
        # 更新历史（不包含财报片段）
        history.append({"role": "user", "content": query})
        history.append({"role": "assistant", "content": answer})
        
        print(f'\nAI 回答:\n{answer}')
        return
    
    # --- 2. Query Rewriting（处理代词、省略）---
    query = rewrite(query, history)
    
    # --- 3. LLM 智能路由 ---
    target_docs = router(query)
    
    # --- 4. 第二步改写 ---
    query = rewrite_for_retrieval(query)
    print(f'改写后用于检索的query:{query}')

    # --- 5. 检索逻辑：支持跨文档分别检索 ---
    all_results = []
    
    if target_docs == [None]:
        # 检索全部文档
        results = test_search(query, mode=mode, top_k=top_k, source_filter=None)
        all_results = results
    elif len(target_docs) == 1:
        # 单个文档
        results = test_search(query, mode=mode, top_k=top_k, source_filter=target_docs[0])
        all_results = results
    else:
        # 多个文档：分别检索每个文档，然后合并
        print(f" 跨文档查询，分别检索 {len(target_docs)} 个文档...")
        for doc in target_docs:
            print(f"\n  正在检索: {doc}")
            results = test_search(query, mode=mode, top_k=top_k, source_filter=doc)
            all_results.extend(results)
        
        # 按分数重新排序并取top_k
        if mode == "hybrid":
            all_results.sort(key=lambda x: x[2], reverse=True)
        elif mode == "dense":
            all_results.sort(key=lambda x: x[2])  # 距离越小越好
        elif mode == "sparse":
            all_results.sort(key=lambda x: x[2], reverse=True)
        
        # all_results = all_results[:top_k]
        print(f"\n 合并后取 top-{len(all_results)} 结果")

    if not all_results:
        print("\n 无法生成回答：未找到相关内容")
        return

    # --- 6. 解析占位符 ---
    print("\n 正在解析表格占位符...")
    all_results = resolve_placeholders(all_results)
    
    retrieved_contexts = []
    for _, text, _, source in all_results:
        # 去掉 .md 后缀
        clean_source = source.replace('.md', '')
        # 拼接成带有来源标识的区块
        context_block = f"【来源文件：{clean_source}】\n{text}"
        retrieved_contexts.append(context_block)
        
    context_text = "\n\n---\n\n".join(retrieved_contexts)

    # 删除历史中旧的财报片段
    history[:] = [msg for msg in history if not (msg.get('role') == 'system' and '【本轮财报片段】' in msg.get('content', ''))]
    
    # 添加新的财报片段和用户问题
    history.append({"role": "system", "content": f"【本轮财报片段】\n{context_text}"})
    history.append({"role": "user", "content": query})
    
    # 直接用 history 调用 LLM
    response = client.chat.completions.create(
        model=MODEL,    
        extra_body={"thinking": {"type": "disabled"}},
        messages=history,
        temperature=0.1
    )
    answer = response.choices[0].message.content
    
    history.append({"role": "assistant", "content": answer})
    
    print('\nAI 回答')
    print(f'AI回答: {answer}')

In [7]:
# init_system()
query = '这家公司的员工规模'
results = test_generation(query, top_k=10)

需要检索
🤖 LLM返回: ['盛和资源控股股份有限公司2019年年度报告.md']
✅ 匹配到文档: ['盛和资源控股股份有限公司2019 年年度报告.md']
 检索优化rewrite: 盛和资源的员工规模是多少？ → 员工规模
改写后用于检索的query:员工规模

 查询问题: 员工规模
 检索模式: hybrid
 返回数量: 10
 混合检索 (BGE向量 + BM25 + RRF)

 检索到 10 条相关片段:
------------------------------------------------------------

[片段 1] ID: 3074 | 来源: 盛和资源控股股份有限公司2019 年年度报告.md
  RRF得分: 0.0325 | 向量相似度: 0.7414
  内容: <html><body><table><tr><td>母公司在职员工的数量</td><td>32</td></tr><tr><td>主要子公司在职员工的数量</td><td>2.014</td></tr><tr><td>在职员工的数量合计</td><td>2,046</td></tr><tr><td>母公司及主要子公司需承担费用的离退休职工 人数</td><td>0</td></tr><tr><td colspan="2">专业构成</td></tr><tr><td>专业构成类别</td><td>专业构成人数</td></tr><tr><td>生产人员</td><td>1,307</td></tr><tr><td>销售人员</td><td>48</td></tr><tr><td>技术人员</td><td>319</td></tr><tr><td>财务人员</td><td>54</td></tr><tr><td>行政人员</td><td>318</td></tr><tr><td>合计</td><td>2,046</td></tr><tr><td colspan="2">教育程度</td></tr><tr><td>教育程度类别</td><td>数量（人）</td></tr><tr><td>博士及硕士研究生</td><td></td></tr><tr><td>本科</td><td>35 163</td></tr><tr><td>大专及以下</td><td>1

In [8]:
history

[{'role': 'user', 'content': '2019年研发投入总额占营业收入'},
 {'role': 'assistant',
  'content': '根据盛和资源控股股份有限公司2019年年度报告，2019年研发投入总额占营业收入比例为 **3.05%**。'},
 {'role': 'user', 'content': '股票代码'},
 {'role': 'assistant',
  'content': '根据盛和资源控股股份有限公司2019年年度报告，公司股票简况显示：\n\n**股票代码：600392**'},
 {'role': 'user', 'content': '子公司'},
 {'role': 'assistant',
  'content': '根据盛和资源控股股份有限公司2019年年度报告，公司的主要子公司情况如下：\n\n### 主要子公司列表\n\n| 子公司名称 | 主要经营地 | 业务性质 | 持股比例（直接/间接） | 取得方式 |\n|-----------|-----------|---------|-------------------|---------|\n| 乐山盛和稀土股份有限公司 | 乐山 | 生产制造 | 99.9999% | - |\n| 四川润和催化新材料股份有限公司 | 乐山 | 生产制造 | 间接38.12% | 非同一控制下企业合并 |\n| 成都市润和盛建石化工程技术有限公司 | 成都 | 咨询行业 | 间接100.00% | 投资设立 |\n| REZEL INTERNATIONAL PTE. LTD. | 新加坡 | 商品流通 | 间接100.00% | 投资设立 |\n| 四川润和功能新材料有限公司 | 绵阳市 | 商品流通 | 间接80.00% | 投资设立 |\n| 润和催化剂沙特有限公司 | 沙特利雅得 | 商品流通 | 间接100.00% | 投资设立 |\n| 德昌盛和新材料科技有限公司 | 德昌县 | 商品流通 | 间接100.00% | 投资设立 |\n| 盛和资源（新加坡）有限公司 | 新加坡 | 商品流通 | 间接100.00% | 投资设立 |\n| 越南稀土有限公司 | 越南 | 生产制造 | 间接90.00% | 非同一控制下企业合并 |\n| 盛和

In [ ]:
# 测试表格检索：查看用于检索的summary和用于回答的HTML
import psycopg
from pgvector.psycopg import register_vector
from rag.database import get_db_connection

# 执行一个会检索表格的查询
test_query = "合诚工程咨询集团的员工规模"

print("=" * 60)
print(f"测试查询: {test_query}")
print("=" * 60)

# 使用混合检索获取结果
from rag import hybrid_search

results = hybrid_search(test_query, top_k=10)

print("\n" + "=" * 60)
print("检索结果分析")
print("=" * 60)

# 连接数据库获取完整信息
conn = get_db_connection()
register_vector(conn)

for doc_id, chunk_text, score, source_file in results:
    print(f"\n{'='*60}")
    print(f"[片段 ID: {doc_id}] RRF得分: {score:.4f}")
    print("=" * 60)
    
    # 从数据库查询该记录的summary和table_id
    cur = conn.execute(
        "SELECT summary, table_id, source_file FROM financial_vectors WHERE id = %s",
        (doc_id,)
    )
    row = cur.fetchone()
    
    if row:
        db_summary, table_id, db_source_file = row
        
        print(f"\n来源文件: {db_source_file}")
        
        if table_id:
            print(f"表格ID: {table_id}")
            print(f"\n【用于检索的Summary】:")
            print("-" * 40)
            print(db_summary if db_summary else "(无summary)")
            
            print(f"\n【用于回答的HTML代码】:")
            print("-" * 40)
            print(chunk_text[:500] if len(chunk_text) > 500 else chunk_text)
            if len(chunk_text) > 500:
                print(f"\n... (共{len(chunk_text)}字符)")
        else:
            print("非表格内容")
            print(f"\n【内容预览】:")
            print("-" * 40)
            print(chunk_text[:300] if len(chunk_text) > 300 else chunk_text)
    else:
        print("未找到数据库记录")

conn.close()

# 验证resolve_placeholders的效果
print("\n\n" + "=" * 60)
print("验证占位符解析")
print("=" * 60)

from rag import resolve_placeholders

resolved_results = resolve_placeholders(results)
print("\n解析前后的对比:")
for i, (doc_id, text, score, _) in enumerate(results[:2]):
    print(f"\n[片段 {i+1}]")
    print(f"解析前: {text[:100]}...")
    resolved_text = resolved_results[i][1]
    print(f"解析后: {resolved_text[:200]}...")